# 🎓 Smart Classroom — High-Precision PyTorch Object Detector (v4)
### Multi-Class: `student` (person folder) + `janitor` (bucket folder)
### Format: Roboflow COCO JSON — Faster R-CNN ResNet-50 FPN — 150 Epochs

**Workflow:**
1. Upload `person/` folder to Drive → `MyDrive/classroom_dataset/person/`
2. Upload `bucket/` folder to Drive → `MyDrive/classroom_dataset/bucket/`
3. Run all cells — training ~20-30 min on T4 GPU
4. Download `best_classroom_detector.pth` + `detector_config.json` from Drive

In [ ]:
# Step 1: Mount Google Drive & Check GPU
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, random, copy
import numpy as np
import cv2
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image

print(f"PyTorch: {torch.__version__}  |  GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Enable GPU -> Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Step 2: Configuration
# ── Upload your two Roboflow folders to Drive exactly here ────────────────
PERSON_DIR  = '/content/drive/MyDrive/classroom_dataset/person'   # student images
BUCKET_DIR  = '/content/drive/MyDrive/classroom_dataset/bucket'   # janitor/bucket images

# Class IDs (matches inference_engine.py)
CLASS_MAP = {
    'person':  1,   # student
    'bucket':  2,   # janitor
    'janitor': 2,
}

# Hyperparameters
NUM_CLASSES  = 3        # 0=background, 1=student, 2=janitor
EPOCHS       = 150
BATCH_SIZE   = 4
LEARNING_RATE= 0.0003
CONF_THRESH  = 0.50
IMG_SIZE     = 800      # Faster R-CNN default input size

OUTPUT_DIR   = '/content/drive/MyDrive/classroom_model_v4'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify folders exist
for d in [PERSON_DIR, BUCKET_DIR]:
    for split in ['train', 'valid']:
        p = os.path.join(d, split)
        if os.path.exists(p):
            imgs = [f for f in os.listdir(p) if f.lower().endswith(('.jpg','.png','.jpeg'))]
            ann  = os.path.exists(os.path.join(p, '_annotations.coco.json'))
            print(f"  {p.split('MyDrive/')[-1]:45s} -> {len(imgs)} images | annotations: {ann}")
        else:
            print(f"  NOT FOUND: {p}")

In [ ]:
# Step 3: COCO JSON Dataset Class (Merged Person + Bucket)

class MergedCOCODataset(Dataset):
    """
    Reads Roboflow COCO JSON format from two dataset folders:
      - person/  (student)
      - bucket/  (janitor)
    Each split folder contains:
      _annotations.coco.json
      *.jpg images
    """
    def __init__(self, splits=('train',)):
        self.records = []   # list of (img_path, [{box, label}])

        for ds_folder, class_name in [(PERSON_DIR, 'person'), (BUCKET_DIR, 'bucket')]:
            class_id = CLASS_MAP[class_name]
            for split in splits:
                split_dir = os.path.join(ds_folder, split)
                ann_path  = os.path.join(split_dir, '_annotations.coco.json')

                if not os.path.exists(ann_path):
                    print(f"  Skipping {split_dir} (no annotations found)")
                    continue

                with open(ann_path) as f:
                    coco = json.load(f)

                # Build id->filename map
                id2file = {img['id']: img['file_name'] for img in coco['images']}

                # Group annotations by image_id
                from collections import defaultdict
                img2anns = defaultdict(list)
                for ann in coco['annotations']:
                    img2anns[ann['image_id']].append(ann)

                for img_id, anns in img2anns.items():
                    img_file = os.path.join(split_dir, id2file[img_id])
                    if not os.path.exists(img_file):
                        continue

                    boxes = []
                    for ann in anns:
                        # COCO box format: [x, y, width, height]
                        x, y, w, h = ann['bbox']
                        x1, y1, x2, y2 = float(x), float(y), float(x+w), float(y+h)
                        if w >= 4 and h >= 4:
                            boxes.append({'x1':x1,'y1':y1,'x2':x2,'y2':y2,'label':class_id})

                    if boxes:
                        self.records.append((img_file, boxes))

                print(f"  [{split.upper():5s}] {class_name:8s}: {sum(1 for r in self.records if any(b['label']==class_id for b in r[1]))} images loaded")

        random.shuffle(self.records)
        print(f"\n  Total merged records: {len(self.records)}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        img_path, ann_list = self.records[idx]

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            raise ValueError(f"Cannot read: {img_path}")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_t   = torch.from_numpy(img_rgb).permute(2,0,1).float() / 255.0

        boxes  = [[a['x1'],a['y1'],a['x2'],a['y2']] for a in ann_list]
        labels = [a['label'] for a in ann_list]

        target = {
            'boxes':    torch.tensor(boxes,  dtype=torch.float32),
            'labels':   torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx])
        }
        return img_t, target

def collate_fn(batch):
    return tuple(zip(*batch))

print("Building merged dataset...")
full_dataset = MergedCOCODataset(splits=('train','valid'))

# 85/15 train-val split
n_val   = max(4, int(len(full_dataset) * 0.15))
n_train = len(full_dataset) - n_val
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

print(f"\nTrain: {len(train_ds)} images | Val: {len(val_ds)} images")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# Step 4: Build Faster R-CNN with ResNet-50 FPN Backbone

def get_model(num_classes):
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model   = fasterrcnn_resnet50_fpn_v2(weights=weights)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = get_model(NUM_CLASSES).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: Faster R-CNN ResNet-50 FPN v2")
print(f"Device: {device}")
print(f"Trainable params: {n_params:,}")
print(f"Classes: background=0 | student=1 | janitor=2")

In [ ]:
# Step 5: Optimizer & Cosine Learning Rate Scheduler

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=5e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

print(f"Optimizer: AdamW  lr={LEARNING_RATE}  weight_decay=5e-4")
print(f"Scheduler: CosineAnnealingLR  T_max={EPOCHS}  eta_min=1e-6")
print(f"Training for {EPOCHS} epochs...")

In [ ]:
# Step 6: Training Loop (150 Epochs)

best_loss   = float('inf')
best_path   = os.path.join(OUTPUT_DIR, 'best_classroom_detector.pth')
loss_log    = []
start_time  = time.time()

print("="*70)
print(f"  Epoch  |  Train Loss  |  Class Loss  |  Box Loss  |  LR       |  Time")
print("="*70)

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    t_loss = t_cls = t_box = 0.0

    for images, targets in train_loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        losses    = sum(v for v in loss_dict.values())
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()

        t_loss += losses.item()
        t_cls  += loss_dict.get('loss_classifier', torch.tensor(0.)).item()
        t_box  += loss_dict.get('loss_box_reg',    torch.tensor(0.)).item()

    scheduler.step()

    n_batches   = max(1, len(train_loader))
    avg_loss    = t_loss / n_batches
    avg_cls     = t_cls  / n_batches
    avg_box     = t_box  / n_batches
    current_lr  = optimizer.param_groups[0]['lr']
    elapsed     = time.time() - start_time
    loss_log.append(avg_loss)

    # Save best
    tag = ""
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), best_path)
        tag = " ⭐"

    if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
        mins = int(elapsed // 60); secs = int(elapsed % 60)
        print(f"  {epoch:5d}  |  {avg_loss:10.4f}  |  {avg_cls:10.4f}  |  {avg_box:8.4f}  |  {current_lr:.6f}  |  {mins}m{secs:02d}s{tag}")

print("="*70)
total_time = time.time() - start_time
print(f"Training complete in {int(total_time//60)}m {int(total_time%60)}s")
print(f"Best loss: {best_loss:.4f}")
print(f"Saved: {best_path}")

In [ ]:
# Step 7: Quick Validation — Count Detection Accuracy

model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

total_imgs = 0
correct_count = 0   # frames where detected count matches true count
total_student_true = 0; total_student_pred = 0
total_janitor_true = 0; total_janitor_pred = 0

with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(device) for img in images]
        preds  = model(images)

        for pred, tgt in zip(preds, targets):
            keep  = pred['scores'] > CONF_THRESH
            pred_labels = pred['labels'][keep].cpu().tolist()
            true_labels = tgt['labels'].tolist()

            n_pred_s = pred_labels.count(1); n_true_s = true_labels.count(1)
            n_pred_j = pred_labels.count(2); n_true_j = true_labels.count(2)

            total_student_true += n_true_s; total_student_pred += n_pred_s
            total_janitor_true += n_true_j; total_janitor_pred += n_pred_j

            if n_pred_s == n_true_s and n_pred_j == n_true_j:
                correct_count += 1
            total_imgs += 1

acc = 100 * correct_count / max(1, total_imgs)
print("="*50)
print(f"Exact Frame Count Accuracy: {acc:.1f}%  ({correct_count}/{total_imgs} frames)")
print(f"Students — True: {total_student_true}  Predicted: {total_student_pred}")
print(f"Janitors — True: {total_janitor_true}  Predicted: {total_janitor_pred}")
print("="*50)

In [ ]:
# Step 8: Save detector_config.json to Drive
cfg = {
    "num_classes": NUM_CLASSES,
    "class_names": ["background", "student", "janitor"],
    "confidence_threshold": CONF_THRESH,
    "model_architecture": "FasterRCNN_ResNet50_FPN_V2",
    "version": 4
}
cfg_path = os.path.join(OUTPUT_DIR, 'detector_config.json')
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print("Files saved to Google Drive:")
print(f"  1. {best_path}")
print(f"  2. {cfg_path}")
print()
print("Download both files and place in:")
print("  edge_app/model/best_classroom_detector.pth")
print("  edge_app/model/detector_config.json")